# IndicMaxxer-2B: Sarvam-1 + our tokenizer fusion + /reasoning & /instruct + IndicMaxxer identity
**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **on**, Secret `HF_TOKEN` attached
(open https://huggingface.co/sarvamai/sarvam-1 once and accept its terms if asked; licence: Sarvam non-commercial).
**Input dataset** needs: `indicmaxxer-48k.json` (our tokenizer), `heldout.json`, `indicmaxxer_ultimate.jsonl`.

Run with **Save Version → Save & Run All (Commit)**. Total ≈ 2 h (data 15 · fusion 10 · training 75 · merge 3 · test 5 min). Each step checks the previous one and stops with a clear message if it failed. Training saves its progress every 15 min.
Output: `/kaggle/working/IndicMaxxer-2B` (merged model + tokenizer) and `/kaggle/working/reports/`.

In [ ]:
# 1) Setup
import os, sys, json, glob, subprocess, random, time
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "peft", "accelerate",
                       "datasets", "tokenizers", "regex", "indic-transliteration"])
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import getpass
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face READ token (hidden): ")
INPUT = os.environ.get("IMX_INPUT", "/kaggle/input")
WORK = os.environ.get("IMX_WORK", "/kaggle/working")
REP = f"{WORK}/reports"
os.makedirs(REP, exist_ok=True)
found = {os.path.basename(p): p for p in glob.glob(f"{INPUT}/**/*", recursive=True) if os.path.isfile(p)}
for f in ["indicmaxxer-48k.json", "heldout.json", "indicmaxxer_ultimate.jsonl"]:
    print(f, "OK" if f in found else "MISSING")
assert "indicmaxxer-48k.json" in found and "heldout.json" in found, "upload our tokenizer + heldout.json"

In [ ]:
%%writefile build_ultimate.py
#!/usr/bin/env python3
"""IndicMaxxer ULTIMATE chat dataset -> ONE jsonl.

Pulls the best open chat/instruction data for Kannada, Hindi, English, Hinglish, Kanglish, cleans it, balances it,
adds our own Codex reasoning + hand-written conversations, and writes a single file:
    data/ultimate/indicmaxxer_ultimate.jsonl
Each line: {id, lang, source, task, mode, messages:[{role,content}], prompt, completion}
  - messages   : canonical chat (last message = assistant target; may carry "think")
  - prompt/completion : already rendered in IndicMaxxer chat format (drop-in for the SFT cell, loss on completion)

Run where `datasets` works (Kaggle / Colab cell):  !python pipeline/build_ultimate.py --hf-token $HF_TOKEN
Every source is optional: a failing source is skipped with a message, never fatal.
"""
import argparse, glob, hashlib, json, os, random, re, time, unicodedata
from collections import Counter, defaultdict

SYSTEM = {
    "kn": "ನೀನು IndicMaxxer: ವೃತ್ತಿಪರ, ಆತ್ಮೀಯ, ನೇರ ಮಾತು. ಸ್ವಲ್ಪ ಹಾಸ್ಯ, ಅತಿಯಾಗಿ ಅಲ್ಲ. ಬಳಕೆದಾರರು ಕನ್ನಡದಲ್ಲಿ ಬರೆದಿದ್ದಾರೆ: ಕನ್ನಡ ಲಿಪಿಯಲ್ಲಿ ಮಾತ್ರ ಉತ್ತರಿಸು.",
    "hi": "तुम IndicMaxxer हो: पेशेवर, गर्मजोशी भरे, सटीक। हल्का सा मज़ाक, ज़्यादा नहीं। उपयोगकर्ता ने हिंदी में लिखा है: सिर्फ़ देवनागरी हिंदी में जवाब दो।",
    "en": "You are IndicMaxxer: professional, warm, crisp. Light dry humour, never overdone. "
          "The user wrote in English: reply in English. Tools: web_search, doc_read, doc_write.",
    "kanglish": "You are IndicMaxxer: professional, warm, crisp. Light dry humour, never overdone. "
                "The user wrote Kanglish (Kannada in English letters): reply in Kanglish only, "
                "not in Kannada script, not in Hindi. Tools: web_search, doc_read, doc_write.",
    "hinglish": "You are IndicMaxxer: professional, warm, crisp. Light dry humour, never overdone. "
                "The user wrote Hinglish (Hindi in English letters): reply in Hinglish only, "
                "not in Devanagari, not in Kannada. Tools: web_search, doc_read, doc_write.",
    "default": "You are IndicMaxxer: professional, warm, crisp. Light dry humour, never overdone. "
               "Reply in the user's language and script. Tools: web_search, doc_read, doc_write.",
}
# target share of the final file (kanglish is scarce online: filled by our data + romanised Kannada)
TARGET = {"kn": 0.30, "hi": 0.20, "en": 0.20, "hinglish": 0.15, "kanglish": 0.15}

# (hf repo, config, split, expected lang or None = detect, cap, task)
HF_SOURCES = [
    ("HuggingFaceTB/smoltalk", "everyday-conversations", "train", "en", 3000, "conversation"),
    ("HuggingFaceTB/smoltalk", "smol-magpie-ultra", "train", "en", 9000, "instruction"),
    ("CohereLabs/aya_dataset", None, "train", None, 30000, "instruction"),          # kn / hi / en rows by language
    ("rishiraj/samvaad-hi-v1", None, "train", None, 8000, "conversation"),
    ("ai4bharat/indic-align", "Anudesh", "train", None, 12000, "instruction"),
    ("ai4bharat/indic-align", "Indic_ShareLlama", "train", None, 12000, "instruction"),
    ("Tensoic/gooftagoo", None, "train", None, 8000, "conversation"),
    ("DSMJ910/hinglish-instruct-10k", None, "train", "hinglish", 8000, "instruction"),
]

BAD = re.compile(r"as an ai language model|i am an ai developed by|openai|chatgpt|मैं एक एआई भाषा मॉडल|"
                 r"llama|meta ai|i cannot browse|knowledge cutoff", re.I)
LANG_NAMES = {"kannada": "kn", "kan": "kn", "kn": "kn", "hindi": "hi", "hin": "hi", "hi": "hi",
              "english": "en", "eng": "en", "en": "en", "hinglish": "hinglish"}
HINGLISH_W = set("hai hain kya nahi nahin mein main kaise kyun bhai yaar toh aur bahut accha achha karo karna "
                 "kar raha rahi tha thi ho hoon batao chahiye kuch abhi sab kal aaj".split())
KANGLISH_W = set("illa ide idhe enu yenu hege heLi heli maadi maadu guru maga beku beda gottu nanna ninna "
                 "nimma aagutte aaytu swalpa thumba tumba olle chennagide yaaru elli yaavaga".split())

def nfc(s):
    return unicodedata.normalize("NFC", (s or "").replace("\r\n", "\n")).strip()

def style(text):
    """kn / hi / en / kanglish / hinglish from script + romanised markers."""
    kn = sum(0x0C80 <= ord(c) <= 0x0CFF for c in text)
    hi = sum(0x0900 <= ord(c) <= 0x097F for c in text)
    lat = sum(c.isascii() and c.isalpha() for c in text)
    tot = kn + hi + lat or 1
    if kn / tot > 0.4:
        return "kn"
    if hi / tot > 0.4:
        return "hi"
    words = set(re.findall(r"[a-zA-Z]+", text.lower()))
    k, h = len(words & KANGLISH_W), len(words & HINGLISH_W)
    if k >= 2 and k > h:
        return "kanglish"
    if h >= 2:
        return "hinglish"
    return "en"

# ------------------------------------------------------------------ schema-adaptive extraction -> list of messages
def to_messages(ex):
    for key in ("messages", "conversations", "conversation", "chat", "dialogue"):
        v = ex.get(key)
        if isinstance(v, list) and v and isinstance(v[0], dict):
            out = []
            for m in v:
                role = (m.get("role") or m.get("from") or m.get("speaker") or "").lower()
                text = m.get("content") or m.get("value") or m.get("text") or ""
                role = {"human": "user", "gpt": "assistant", "bot": "assistant", "model": "assistant"}.get(role, role)
                if role in ("user", "assistant", "system"):
                    out.append({"role": role, "content": nfc(text)})
            return out
    v = ex.get("interactions")                         # indic-align style: [[user, assistant], ...]
    if isinstance(v, list) and v:
        out = []
        for pair in v:
            if isinstance(pair, (list, tuple)) and len(pair) >= 2:
                out += [{"role": "user", "content": nfc(pair[0])}, {"role": "assistant", "content": nfc(pair[1])}]
        return out
    for q, a in (("inputs", "targets"), ("instruction", "output"), ("prompt", "response"), ("question", "answer"),
                 ("input", "output"), ("prompt", "completion"), ("query", "response")):
        if ex.get(q) and ex.get(a):
            user = nfc(ex[q])
            if q == "instruction" and ex.get("input"):
                user += "\n\n" + nfc(ex["input"])
            return [{"role": "user", "content": user}, {"role": "assistant", "content": nfc(ex[a])}]
    return []

def clean_dialog(msgs, max_turns=6):
    msgs = [m for m in msgs if m["role"] != "system" and m["content"]]
    while msgs and msgs[0]["role"] != "user":
        msgs = msgs[1:]
    out = []
    for m in msgs:                                      # enforce strict user/assistant alternation
        want = "user" if len(out) % 2 == 0 else "assistant"
        if m["role"] != want:
            break
        out.append(m)
    if len(out) % 2:
        out = out[:-1]
    out = out[-2 * max_turns:]
    return out if len(out) >= 2 else []

def ok(msgs, lang):
    for m in msgs:
        c = m["content"]
        if len(c) > (1800 if m["role"] == "assistant" else 1500) or BAD.search(c):
            return False
        if m["role"] == "assistant" and len(c) < 2:
            return False
        if re.search(r"(.{6,}?)\1{4,}", c):              # degenerate repetition
            return False
    last_user, last_bot = msgs[-2]["content"], msgs[-1]["content"]
    su, sb = style(last_user), style(last_bot)
    if lang in ("kn", "hi"):
        return su == lang and sb == lang                # same script both sides
    if lang == "en":
        return su == "en" and sb == "en"
    return su == lang and sb == lang                    # romanised: reply must stay romanised, same style

# ------------------------------------------------------------------ rendering (IndicMaxxer chat format)
def render(msgs, lang, rng, mode=None, think=""):
    mode = mode or "no_think"
    flag = rng.choices([mode, "auto"], [0.93, 0.07])[0]
    parts = [f"<|im_start|>system\n{SYSTEM.get(lang, SYSTEM['default'])}<|im_end|>\n"] if rng.random() < 0.7 else []
    for i, m in enumerate(msgs[:-1]):
        c = m["content"]
        if i == len(msgs) - 2 and flag != "auto":
            c += " /think" if mode == "think" else " /no_think"
        parts.append(f"<|im_start|>{m['role']}\n{c}<|im_end|>\n")
    prompt = "".join(parts) + "<|im_start|>assistant\n"
    completion = f"<think>\n{think}\n</think>\n\n{msgs[-1]['content']}<|im_end|>\n" if think \
        else f"<think>\n\n</think>\n\n{msgs[-1]['content']}<|im_end|>\n"
    return prompt, completion

# ------------------------------------------------------------------ Kanglish from Kannada (casual romanisation)
def kanglish_of(text):
    try:
        from indic_transliteration import sanscript
        from indic_transliteration.sanscript import transliterate
    except ImportError:
        return None
    r = transliterate(text, sanscript.KANNADA, sanscript.ITRANS)
    r = re.sub(r"[\^~]", "", r).replace("RRi", "ru").replace("RRI", "ru")
    r = r.replace("aa", "aa").replace("A", "aa").replace("I", "ii").replace("U", "uu").replace("E", "e").replace("O", "o")
    r = re.sub(r"[.]{1}N", "n", r).replace(".a", "").replace("M", "m").replace("H", "h")
    return r.lower()

# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--hf-token", default=os.environ.get("HF_TOKEN"))
    ap.add_argument("--out", default="data/ultimate/indicmaxxer_ultimate.jsonl")
    ap.add_argument("--total", type=int, default=60000, help="approx rows in the final file")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--max-minutes", type=float, default=2.5, help="time cap per source")
    ap.add_argument("--own", nargs="*", default=[], help="existing ultimate jsonl(s) to reuse as our own data")
    ap.add_argument("--target", default=None, help='JSON language shares, e.g. {"en":0.5,"kn":0.2,...}')
    args = ap.parse_args()
    if not args.hf_token:
        try:
            from kaggle_secrets import UserSecretsClient
            args.hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if args.target:
        TARGET.clear(); TARGET.update(json.loads(args.target))
    rng = random.Random(args.seed)
    pool = defaultdict(list)                            # lang -> [record]
    seen, report = set(), Counter()

    def add(msgs, lang, source, task, mode="no_think", think=""):
        key = hashlib.md5((lang + mode + msgs[-2]["content"][:300]).encode()).hexdigest()
        if key in seen:
            report["dup"] += 1
            return
        seen.add(key)
        pool[lang].append({"lang": lang, "source": source, "task": task, "mode": mode, "messages": msgs, "think": think})

    # 1) our own data (highest priority, always kept)
    own = 0
    for path in sorted(glob.glob("data/sft/*train*.jsonl")) + sorted(glob.glob("data/convo/*.jsonl")):
        for line in open(path, encoding="utf-8"):
            if not line.strip():
                continue
            r = json.loads(line)
            if "messages" in r:                         # hand-written convos: one record per assistant turn
                ms = r["messages"]
                for i in range(1, len(ms), 2):
                    add([{"role": m["role"], "content": m["content"]} for m in ms[:i + 1]], r["lang"],
                        "indicmaxxer/convo", "conversation")
                    own += 1
            elif "prompt" in r and r.get("task") == "reasoning":   # codex reasoning (already rendered)
                user = r["prompt"].rsplit("<|im_start|>user\n", 1)[1].split("<|im_end|>")[0]
                user = user.replace(" /think", "").replace(" /no_think", "")
                comp = r["completion"]
                think = comp.split("<think>\n", 1)[1].split("\n</think>")[0].strip() if "<think>" in comp else ""
                ans = comp.split("</think>\n\n", 1)[-1].replace("<|im_end|>\n", "").strip()
                add([{"role": "user", "content": user}, {"role": "assistant", "content": ans}], r["lang"],
                    "indicmaxxer/codex", "reasoning", r["mode"], think if r["mode"] == "think" else "")
                own += 1
    for path in args.own:                               # a previous ultimate file: reuse its own-data rows
        for line in open(path, encoding="utf-8"):
            r = json.loads(line)
            if r.get("source", "").startswith("indicmaxxer"):
                last = r["messages"][-1]
                ms = r["messages"][:-1] + [{"role": "assistant", "content": last["content"]}]
                add(ms, r["lang"], r["source"], r["task"], r["mode"], last.get("think", ""))
                own += 1
    print("own records:", own)

    # 2) Hugging Face sources
    try:
        from datasets import load_dataset
    except ImportError:
        load_dataset = None
        print("`datasets` not installed: only our own data will be written")
    for repo, cfg, split, lang_fixed, cap, task in (HF_SOURCES if load_dataset else []):
        name = f"{repo}" + (f"/{cfg}" if cfg else "")
        kept = scanned = 0
        try:
            ds = load_dataset(repo, cfg, split=split, streaming=True, token=args.hf_token)
            t_src = time.time()
            for ex in ds:
                scanned += 1
                if kept >= cap or scanned > cap * 4 or time.time() - t_src > args.max_minutes * 60:
                    break
                if scanned % 5000 == 0:
                    print(f"  {name}: scanned {scanned}, kept {kept}", flush=True)
                lang_col = str(ex.get("language") or ex.get("lang") or ex.get("language_code") or "").lower()
                msgs = clean_dialog(to_messages(ex))
                if not msgs:
                    report[f"{name}:unparsed"] += 1
                    continue
                lang = lang_fixed or LANG_NAMES.get(lang_col) or style(msgs[-2]["content"])
                if lang not in TARGET:
                    continue
                if not ok(msgs, lang):
                    report[f"{name}:filtered"] += 1
                    continue
                add(msgs, lang, name, task)
                kept += 1
            print(f"{name}: kept {kept} / scanned {scanned}")
        except Exception as e:
            print(f"{name}: SKIPPED ({type(e).__name__}: {str(e)[:120]})")

    # 3) Kanglish top-up: romanise Kannada conversations (Kanglish is scarce online)
    need = int(args.total * TARGET["kanglish"]) - len(pool["kanglish"])
    if need > 0 and pool["kn"]:
        made = 0
        for r in rng.sample(pool["kn"], min(len(pool["kn"]), need * 2)):
            if made >= need or r["task"] == "reasoning":
                continue
            ms = [{"role": m["role"], "content": kanglish_of(m["content"])} for m in r["messages"]]
            if any(m["content"] is None or len(m["content"]) < 2 for m in ms):
                continue
            add(ms, "kanglish", r["source"] + "+romanised", r["task"])
            made += 1
        print("kanglish romanised from kannada:", made)

    # 4) balance to target shares (own data always kept first)
    final = []
    for lang, share in TARGET.items():
        recs = pool[lang]
        mine = [r for r in recs if r["source"].startswith("indicmaxxer")]
        rest = [r for r in recs if not r["source"].startswith("indicmaxxer")]
        rng.shuffle(rest)
        quota = max(int(args.total * share) - len(mine), 0)
        final += mine + rest[:quota]
    rng.shuffle(final)

    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    stats = Counter()
    with open(args.out, "w", encoding="utf-8") as f:
        for i, r in enumerate(final):
            prompt, completion = render(r["messages"], r["lang"], rng, r["mode"], r["think"])
            msgs = r["messages"][:-1] + [dict(r["messages"][-1], **({"think": r["think"]} if r["think"] else {}))]
            f.write(json.dumps({"id": f"imx-{i:06d}", "lang": r["lang"], "source": r["source"], "task": r["task"],
                                "mode": r["mode"], "messages": msgs, "prompt": prompt, "completion": completion},
                               ensure_ascii=False) + "\n")
            stats[(r["lang"], r["source"])] += 1
    by_lang = Counter(r["lang"] for r in final)
    print(f"\nwrote {len(final)} rows -> {args.out}")
    print("by language:", dict(by_lang))
    print("by source:")
    for (lang, src), n in sorted(stats.items()):
        print(f"  {lang:9s} {src:45s} {n}")
    print("dropped:", dict(report.most_common(12)))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile imx2b_fuse.py
#!/usr/bin/env python3
"""IndicMaxxer-2B step 1: fuse our tokenizer into Sarvam-1's, add chat/think tokens, resize + initialise embeddings.

Fusion rule (only add a token where ours is cheaper):
  for frequent words in our 5 styles (kn / hi / en / Kanglish / Hinglish), if Sarvam splits the word into >= 3 pieces
  and our 48K tokenizer uses fewer, add " word" as one new token. New embedding = mean of the Sarvam pieces it replaces.
Then measure tokens/word per style BEFORE vs AFTER on held-out text; if a style got worse, its tokens are dropped.

Usage: python imx2b_fuse.py fuse_config.json
"""
import json, os, sys, time
from collections import Counter
import regex
import torch
from tokenizers import Tokenizer
from transformers import AddedToken, AutoModelForCausalLM, AutoTokenizer

C = json.load(open(sys.argv[1]))
HF = os.environ.get("HF_TOKEN") or None
STYLES = ["kn", "hi", "en", "kanglish", "hinglish"]
WORD = regex.compile(r"[\p{L}\p{M}]+")
SPECIALS = ["<|im_start|>", "<|im_end|>", "<think>", "</think>"]
t0 = time.time()

tok = AutoTokenizer.from_pretrained(C["base"], token=HF)
ours = Tokenizer.from_file(C["our_tok"])
held = json.load(open(C["heldout"], encoding="utf-8"))
extra = {s: [] for s in STYLES}                      # more romanised text from the chat dataset, if given
if C.get("extra_jsonl") and os.path.exists(C["extra_jsonl"]):
    for line in open(C["extra_jsonl"], encoding="utf-8"):
        r = json.loads(line)
        if r.get("lang") in extra:
            extra[r["lang"]] += [m["content"] for m in r.get("messages", [])]

def pieces(t, w):
    """How many tokens tokenizer t spends on w in the middle of a sentence."""
    return len(t.encode("a " + w, add_special_tokens=False)) - len(t.encode("a", add_special_tokens=False))

def tpw(t, docs):
    ids = words = 0
    for d in docs:
        ids += len(t.encode(d, add_special_tokens=False))
        words += len(WORD.findall(d))
    return round(ids / max(words, 1), 3)

eval_docs = {s: held.get(s, [])[-120:] for s in STYLES}
cand_docs = {s: held.get(s, [])[:-120] + extra[s] for s in STYLES}
before = {s: tpw(tok, eval_docs[s]) for s in STYLES if eval_docs[s]}
print("Sarvam tokens/word BEFORE:", before, flush=True)

# ---------------------------------------------------------------- pick fusion tokens
chosen = {}
for s in STYLES:
    freq = Counter()
    for d in cand_docs[s]:
        for w in WORD.findall(d):
            freq[w.lower() if s in ("en", "kanglish", "hinglish") else w] += 1
    scored = []
    for w, f in freq.most_common(C.get("scan_top", 20000)):
        if f < C["min_freq"]:
            break                                     # most_common is sorted: the rest are rarer
        if len(w) < 3:
            continue
        ps = pieces(tok, w)
        if ps < 3:
            continue
        po = len(ours.encode(" " + w).ids)
        if po >= ps:
            continue                                  # ours is not cheaper: not fused
        scored.append((f * (ps - 1), w, ps, po))
    scored.sort(reverse=True)
    chosen[s] = scored[:C["quota"].get(s, 0)]
    print(f"{s}: {len(scored)} candidates, keeping {len(chosen[s])}; top:",
          [(w, ps, po) for _, w, ps, po in chosen[s][:8]], flush=True)

def apply(selection):
    t = AutoTokenizer.from_pretrained(C["base"], token=HF)
    words = sorted({w for s in selection for _, w, _, _ in selection[s]})
    t.add_tokens([AddedToken(" " + w, single_word=False, lstrip=False, rstrip=False, normalized=False) for w in words])
    return t, words

tok2, words = apply(chosen)
after = {s: tpw(tok2, eval_docs[s]) for s in before}
print("tokens/word AFTER fusion:", after, flush=True)
worse = [s for s in after if after[s] > before[s] + 1e-9]
if worse:                                             # never pay a tax increase: drop styles that got worse
    print("dropping fusion for styles that got worse:", worse)
    for s in worse:
        chosen[s] = []
    tok2, words = apply(chosen)
    after = {s: tpw(tok2, eval_docs[s]) for s in before}
    print("tokens/word AFTER (final):", after)
tok2.add_special_tokens({"additional_special_tokens": SPECIALS})
tok2.chat_template = C["chat_template"]
if tok2.pad_token is None:
    tok2.pad_token = tok2.eos_token

# ---------------------------------------------------------------- model: resize + init
try:
    model = AutoModelForCausalLM.from_pretrained(C["base"], dtype=torch.float16, token=HF)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(C["base"], torch_dtype=torch.float16, token=HF)
old_n = model.get_input_embeddings().weight.shape[0]
model.resize_token_embeddings(len(tok2), pad_to_multiple_of=64)
emb = model.get_input_embeddings().weight.data
head = model.get_output_embeddings().weight.data if model.get_output_embeddings() is not None else None
tied = head is None or head.data_ptr() == emb.data_ptr()
with torch.no_grad():
    mean_e = emb[:old_n].float().mean(0)
    mean_h = head[:old_n].float().mean(0) if not tied else None
    for w in words:
        nid = tok2.convert_tokens_to_ids(" " + w)
        src = tok.encode("a " + w, add_special_tokens=False)[len(tok.encode("a", add_special_tokens=False)):]
        emb[nid] = emb[src].float().mean(0).to(emb.dtype)
        if not tied:
            head[nid] = head[src].float().mean(0).to(head.dtype)
    for sp in SPECIALS:
        nid = tok2.convert_tokens_to_ids(sp)
        emb[nid] = (mean_e + 0.01 * torch.randn_like(mean_e)).to(emb.dtype)
        if not tied:
            head[nid] = (mean_h + 0.01 * torch.randn_like(mean_h)).to(head.dtype)
os.makedirs(C["out_dir"], exist_ok=True)
model.save_pretrained(C["out_dir"], safe_serialization=True)
tok2.save_pretrained(C["out_dir"])
report = {"before": before, "after": after,
          "saving_pct": {s: round(100 * (before[s] - after[s]) / before[s], 1) for s in after},
          "fused_tokens": len(words), "per_style": {s: len(v) for s, v in chosen.items()},
          "vocab_old": old_n, "vocab_new": len(tok2), "tied_embeddings": tied,
          "extra_params_M": round((model.get_input_embeddings().weight.shape[0] - old_n) * emb.shape[1]
                                  * (1 if tied else 2) / 1e6, 2),
          "examples": {s: [w for _, w, _, _ in chosen[s][:15]] for s in chosen}}
json.dump(report, open(C["report"], "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps({k: v for k, v in report.items() if k != "examples"}, indent=1))
print(f"fused model saved to {C['out_dir']} in {(time.time() - t0) / 60:.1f} min")


In [ ]:
%%writefile imx2b_train.py
#!/usr/bin/env python3
"""IndicMaxxer-2B step 2: LoRA fine-tune of the fused Sarvam-1 (fp16 autocast on T4).

Trainable: LoRA on every attention/MLP projection + the full embedding (and output head if untied) in fp32, so fused
Kanglish/Hinglish tokens and chat/think tokens learn. Loss only on the assistant reply.
Saves ONLY the trainable tensors (LoRA + embeddings) to `state_path` every `ckpt_every_min` and at the end, so a crash
still leaves a usable checkpoint. Merging into a full model is a separate step (imx2b_merge.py).

Usage: python imx2b_train.py train_config.json                                  (1 GPU, default, most robust)
       torchrun --standalone --nproc_per_node=2 imx2b_train.py train_config.json (2 GPUs)
"""
import json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
import torch.distributed as dist
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

C = json.load(open(sys.argv[1]))
WORLD, RANK = int(os.environ.get("WORLD_SIZE", 1)), int(os.environ.get("RANK", 0))
LOCAL = int(os.environ.get("LOCAL_RANK", 0))
if WORLD > 1:
    dist.init_process_group("nccl")
torch.cuda.set_device(LOCAL)
DEV = torch.device("cuda", LOCAL)
MAIN = RANK == 0
def log(*a):
    if MAIN:
        print(*a, flush=True)

def load_model(path):
    try:
        return AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16)

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
class NewRows(torch.nn.Module):
    """Trainable fp32 rows ONLY for the tokens we added (fused words + chat/think tokens).
    The original 68K embedding/head rows stay frozen in fp16: saves ~5 GB of optimizer memory on a T4."""
    def __init__(self, model, old_n, new_n):
        super().__init__()
        e, h = model.get_input_embeddings(), model.get_output_embeddings()
        self.old_n, self.new_n = old_n, new_n
        self.tied = h is None or h.weight is e.weight
        self.e = torch.nn.Parameter(e.weight.data[old_n:new_n].float().clone())
        self.h = None if self.tied else torch.nn.Parameter(h.weight.data[old_n:new_n].float().clone())
        e.register_forward_hook(self._emb_hook)
        if h is not None:
            h.register_forward_hook(self._head_hook)
    def _emb_hook(self, mod, inp, out):
        ids = inp[0]
        m = ids >= self.old_n
        if not m.any():
            return out
        out = out.clone()
        out[m] = self.e[(ids[m] - self.old_n).clamp(max=self.new_n - self.old_n - 1)].to(out.dtype)
        return out
    def _head_hook(self, mod, inp, out):
        w = self.e if self.tied else self.h
        new = (inp[0].to(w.dtype) @ w.t()).to(out.dtype)
        return torch.cat([out[..., :self.old_n], new, out[..., self.new_n:]], dim=-1)

def build(path, r, old_n, new_n):
    """Fused base (frozen fp16) + LoRA (fp32) + trainable new-token rows (fp32)."""
    m = load_model(path)
    m.config.use_cache = False
    m = get_peft_model(m, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                     target_modules=LORA_TARGETS))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    rows = NewRows(m, old_n, new_n)
    return m, rows

if __name__ == "__main__":
    tok = AutoTokenizer.from_pretrained(C["model_dir"])
    OLD_N, NEW_N = C["old_vocab"], len(tok)
    model, rows = build(C["model_dir"], C["lora_r"], OLD_N, NEW_N)
    tied = rows.tied
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.to(DEV); rows.to(DEV)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad) + sum(p.numel() for p in rows.parameters())
    log(f"trainable {n_train / 1e6:.0f}M params (LoRA + {NEW_N - OLD_N} new token rows) | GPUs {WORLD}")

    # ------------------------------------------------------------ data
    data_rows = [json.loads(l) for l in open(C["data"], encoding="utf-8")]
    random.Random(0).shuffle(data_rows)
    data_rows = data_rows[RANK::WORLD]
    L, ex = C["max_len"], []
    for r in data_rows:
        p = tok.encode(r["prompt"], add_special_tokens=False)
        c = tok.encode(r["completion"], add_special_tokens=False)
        if not c or len(c) >= L:
            continue
        p = p[-(L - len(c)):]
        ex.append((p + c, [-100] * len(p) + c))
    log(f"examples per GPU: {len(ex)} | avg length {sum(len(e[0]) for e in ex) / max(len(ex), 1):.0f} tokens")
    B = C["micro_b"]
    def batches(epoch):
        rng = random.Random(epoch * 1000 + RANK)
        idx = list(range(len(ex)))
        rng.shuffle(idx)
        groups = [sorted(idx[i:i + B * 64], key=lambda j: len(ex[j][0])) for i in range(0, len(idx), B * 64)]
        bs = [g[i:i + B] for g in groups for i in range(0, len(g), B)]
        rng.shuffle(bs)
        for b in bs:
            m = max(len(ex[j][0]) for j in b)
            x = torch.full((len(b), m), tok.pad_token_id, dtype=torch.long)
            y = torch.full((len(b), m), -100, dtype=torch.long)
            a = torch.zeros((len(b), m), dtype=torch.long)
            for k, j in enumerate(b):
                ids, lab = ex[j]
                x[k, :len(ids)] = torch.tensor(ids); y[k, :len(lab)] = torch.tensor(lab); a[k, :len(ids)] = 1
            yield x.to(DEV), y.to(DEV), a.to(DEV)

    # ------------------------------------------------------------ optimiser
    lora_p = [p for n, p in model.named_parameters() if p.requires_grad]
    emb_p = list(rows.parameters())
    opt = torch.optim.AdamW([{"params": lora_p, "lr": C["lr"], "weight_decay": 0.0},
                             {"params": emb_p, "lr": C["emb_lr"], "weight_decay": 0.0}], betas=(0.9, 0.95))
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda")
    net = torch.nn.parallel.DistributedDataParallel(model, device_ids=[LOCAL], static_graph=True) if WORLD > 1 else model

    def save_state(tag):
        if not MAIN:
            return
        sd = {n: p.detach().to("cpu", torch.float16) for n, p in model.named_parameters() if p.requires_grad}
        nr = {k: v.detach().to("cpu", torch.float32) for k, v in rows.state_dict().items()}
        tmp = C["state_path"] + ".tmp"
        torch.save({"state": sd, "new_rows": nr, "old_n": OLD_N, "new_n": NEW_N, "lora_r": C["lora_r"],
                    "tied": tied, "step": step, "tokens": toks}, tmp)
        os.replace(tmp, C["state_path"])
        log(f"saved trainable state ({tag}) at step {step}")

    A, budget = C["accum"], C["minutes"] * 60
    step, t0, toks, epoch, nan_skips, last_ck = 0, time.time(), 0, 0, 0, time.time()
    it = batches(epoch)
    log(f"micro {B} x accum {A} x {WORLD} GPU(s), max_len {L}, budget {C['minutes']} min. first step compiles kernels...")
    while True:
        el = time.time() - t0
        stop = torch.tensor([1.0 if el >= budget else 0.0], device=DEV)
        if WORLD > 1:
            dist.all_reduce(stop)
        if stop.item() > 0:
            break
        frac = el / budget
        mult = min(1.0, (step + 1) / max(1, C["warmup"])) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(frac, 1.0))))
        for g, b in zip(opt.param_groups, base_lrs):
            g["lr"] = b * mult
        tot = 0.0
        for a in range(A):
            try:
                x, y, am = next(it)
            except StopIteration:
                epoch += 1
                log(f"--- epoch {epoch} complete ---")
                it = batches(epoch)
                x, y, am = next(it)
            ctx = net.no_sync() if (WORLD > 1 and a < A - 1) else torch.enable_grad()
            with ctx:
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = net(input_ids=x, attention_mask=am).logits
                    loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1),
                                           ignore_index=-100) / A
                scaler.scale(loss).backward()
            tot += float(loss.detach())
            toks += int(am.sum()) * WORLD
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(lora_p + emb_p, 1.0)
        if not math.isfinite(tot):                     # fp16 overflow: skip this step, keep going
            nan_skips += 1
            opt.zero_grad(set_to_none=True)
            scaler.update(scaler.get_scale() / 2)
            if nan_skips > 50:
                log("too many non-finite losses: stopping early (try lower lr)")
                break
            continue
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        step += 1
        if step == 1:
            log(f"first step done in {time.time() - t0:.0f}s")
        if step % C["log_every"] == 0 and MAIN:
            now = time.time()
            print({"step": step, "loss": round(tot, 4), "grad_norm": round(float(gn), 3), "lr_mult": round(mult, 3),
                   "epoch": epoch, "min": round((now - t0) / 60, 1), "tokens_M": round(toks / 1e6, 2),
                   "tok_per_s": int(toks / (now - t0)), "nan_skips": nan_skips}, flush=True)
        if time.time() - last_ck > C["ckpt_every_min"] * 60:
            save_state("periodic")
            last_ck = time.time()
    save_state("final")
    log(f"training done: {step} steps, {toks / 1e6:.2f}M tokens, {(time.time() - t0) / 60:.1f} min, epochs {epoch}")
    if WORLD > 1:
        dist.barrier()
        dist.destroy_process_group()


In [ ]:
%%writefile imx2b_merge.py
#!/usr/bin/env python3
"""IndicMaxxer-2B step 3: merge LoRA + trained new-token rows into the fused base -> full fp16 model.
Usage: python imx2b_merge.py train_config.json"""
import json, os, sys
import torch
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from imx2b_train import load_model, LORA_TARGETS

C = json.load(open(sys.argv[1]))
ck = torch.load(C["state_path"], map_location="cpu")
old_n, new_n, r = ck["old_n"], ck["new_n"], ck["lora_r"]
base = load_model(C["model_dir"])
with torch.no_grad():                                    # write trained rows into the real embedding / head
    e, h = base.get_input_embeddings(), base.get_output_embeddings()
    e.weight[old_n:new_n] = ck["new_rows"]["e"].to(e.weight.dtype)
    if "h" in ck["new_rows"] and h is not None and h.weight is not e.weight:
        h.weight[old_n:new_n] = ck["new_rows"]["h"].to(h.weight.dtype)
model = get_peft_model(base, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                        target_modules=LORA_TARGETS))
params = dict(model.named_parameters())
missing = [n for n in ck["state"] if n not in params]
assert not missing, f"LoRA names not found: {missing[:5]}"
with torch.no_grad():
    for n, t in ck["state"].items():
        params[n].copy_(t.to(params[n].dtype))
merged = model.merge_and_unload().half()
merged.config.use_cache = True
os.makedirs(C["out_dir"], exist_ok=True)
merged.save_pretrained(C["out_dir"], safe_serialization=True)
AutoTokenizer.from_pretrained(C["model_dir"]).save_pretrained(C["out_dir"])
json.dump({"step": ck["step"], "tokens": ck["tokens"]}, open(os.path.join(C["out_dir"], "train_stats.json"), "w"))
print(f"IndicMaxxer-2B saved to {C['out_dir']} (step {ck['step']}, {ck['tokens'] / 1e6:.2f}M tokens)")


In [ ]:
# 4) Config: the only cell to touch
BASE = "sarvamai/sarvam-1"
TRAIN_MIN = 75                         # LoRA training time on 2 x T4
TOTAL = 24000                          # SFT examples (SmolTalk-heavy, as decided)
MIX = {"en": 0.45, "kn": 0.22, "hi": 0.11, "kanglish": 0.11, "hinglish": 0.11}
FUSE_QUOTA = {"kanglish": 1500, "hinglish": 1500, "kn": 600, "hi": 300, "en": 0}   # max new tokens per style
CHAT_TEMPLATE = ("{% for m in messages %}<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n{% endfor %}"
                 "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}")
FUSED, OUT = f"{WORK}/sarvam_fused", f"{WORK}/IndicMaxxer-2B"
STATE = f"{WORK}/imx2b_trainable_state.pt"   # LoRA + embeddings, saved every 15 min (survives a crash)
USE_BOTH_GPUS = False                        # True = ~2x faster but less robust; flip only if 1 GPU works

def run(cmd):
    """Run a script, stream its output live, and stop the notebook with a clear message if it fails."""
    import subprocess
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError("step failed (see the error above). Last lines: " + " | ".join(t.strip() for t in tail[-15:]))
print("ok")

In [ ]:
# 5) Data: ultimate mix (SmolTalk + Aya + IndicAlign + Hinglish sets + our chats + Codex native reasoning)
#    rendered in IndicMaxxer-2B chat format with /reasoning and /instruct, plus identity rows
ULT = f"{WORK}/ultimate_2b.jsonl"
if not os.path.exists(ULT):
    t0 = time.time()
    cmd = [sys.executable, "build_ultimate.py", "--out", ULT, "--total", str(TOTAL), "--target", json.dumps(MIX),
           "--max-minutes", "2"]
    if "indicmaxxer_ultimate.jsonl" in found:
        cmd += ["--own", found["indicmaxxer_ultimate.jsonl"]]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3500:], r.stderr[-1200:], f"\n{(time.time() - t0) / 60:.1f} min")

SYS = {
    "kn": "ನೀನು IndicMaxxer, Sarvam-1 ಆಧಾರಿತ ಭಾರತೀಯ ಭಾಷಾ ಸಹಾಯಕ: ವೃತ್ತಿಪರ, ಆತ್ಮೀಯ, ನೇರ, ಸ್ವಲ್ಪ ಹಾಸ್ಯ. ಕನ್ನಡ ಲಿಪಿಯಲ್ಲೇ ಉತ್ತರಿಸು.",
    "hi": "तुम IndicMaxxer हो, Sarvam-1 पर बना भारतीय भाषा सहायक: पेशेवर, गर्मजोशी भरे, सटीक, हल्का मज़ाक। देवनागरी हिंदी में ही जवाब दो।",
    "en": "You are IndicMaxxer, an Indian-language assistant built on Sarvam-1: professional, warm, crisp, light dry humour. Reply in English.",
    "kanglish": "You are IndicMaxxer, an Indian-language assistant built on Sarvam-1. The user writes Kanglish (Kannada in English letters): reply in Kanglish only.",
    "hinglish": "You are IndicMaxxer, an Indian-language assistant built on Sarvam-1. The user writes Hinglish (Hindi in English letters): reply in Hinglish only.",
}
IDENT = {
    "en": (["who are you?", "what's your name?", "what can you do?", "are you ChatGPT?", "which model are you?"],
           "I'm IndicMaxxer, an Indian-language assistant built on Sarvam-1. I chat in Kannada, Hindi, English, Kanglish and Hinglish, reason step by step with /reasoning, answer directly with /instruct, and can search the web, read and write documents, and read scanned text."),
    "kn": (["ನೀನು ಯಾರು?", "ನಿನ್ನ ಹೆಸರೇನು?", "ನೀನು ಏನು ಮಾಡಬಲ್ಲೆ?", "ನೀನು ChatGPT ನಾ?"],
           "ನಾನು IndicMaxxer, Sarvam-1 ಆಧಾರಿತ ಭಾರತೀಯ ಭಾಷಾ ಸಹಾಯಕ. ಕನ್ನಡ, ಹಿಂದಿ, ಇಂಗ್ಲಿಷ್, ಕಂಗ್ಲಿಷ್, ಹಿಂಗ್ಲಿಷ್‌ನಲ್ಲಿ ಮಾತಾಡುತ್ತೇನೆ; /reasoning ಹಾಕಿದರೆ ಕನ್ನಡದಲ್ಲೇ ಹಂತ ಹಂತವಾಗಿ ಯೋಚಿಸುತ್ತೇನೆ, /instruct ಹಾಕಿದರೆ ನೇರ ಉತ್ತರ ಕೊಡುತ್ತೇನೆ."),
    "hi": (["तुम कौन हो?", "तुम्हारा नाम क्या है?", "तुम क्या कर सकते हो?", "क्या तुम ChatGPT हो?"],
           "मैं IndicMaxxer हूँ, Sarvam-1 पर बना भारतीय भाषा सहायक। कन्नड़, हिंदी, अंग्रेज़ी, कंग्लिश और हिंग्लिश में बात करता हूँ; /reasoning पर हिंदी में ही कदम-दर-कदम सोचता हूँ, /instruct पर सीधा जवाब देता हूँ।"),
    "kanglish": (["neenu yaaru?", "ninna hesaru enu?", "neenu enu maadtiya?", "neenu ChatGPT na?"],
                 "Naanu IndicMaxxer, Sarvam-1 mele build aagiro Indian language assistant. Kannada, Hindi, English, Kanglish, Hinglish alli maathadtini; /reasoning haakidre step by step yochistini, /instruct haakidre direct answer kodtini."),
    "hinglish": (["tum kaun ho?", "tumhara naam kya hai?", "tum kya kar sakte ho?", "kya tum ChatGPT ho?"],
                 "Main IndicMaxxer hoon, Sarvam-1 par bana Indian language assistant. Kannada, Hindi, English, Kanglish aur Hinglish mein baat karta hoon; /reasoning pe step by step sochta hoon, /instruct pe seedha jawab deta hoon."),
}
rng = random.Random(0)
def render(msgs, lang, mode, think=""):
    flag = mode if rng.random() < 0.95 else None                       # 5%: no flag, model decides
    parts = [f"<|im_start|>system\n{SYS.get(lang, SYS['en'])}<|im_end|>\n"] if rng.random() < 0.8 else []
    for i, m in enumerate(msgs[:-1]):
        c = m["content"]
        if i == len(msgs) - 2 and flag:
            c += " /reasoning" if flag == "reasoning" else " /instruct"
        parts.append(f"<|im_start|>{m['role']}\n{c}<|im_end|>\n")
    prompt = "".join(parts) + "<|im_start|>assistant\n"
    completion = (f"<think>\n{think}\n</think>\n\n" if think else "<think>\n\n</think>\n\n") + msgs[-1]["content"] + "<|im_end|>"
    return {"prompt": prompt, "completion": completion, "lang": lang, "mode": mode}

out, cnt = [], {}
for line in open(ULT, encoding="utf-8"):
    r = json.loads(line)
    last = r["messages"][-1]
    think = last.get("think", "")
    mode = "reasoning" if think else "instruct"
    out.append(render(r["messages"], r["lang"], mode, think))
    cnt[(r["lang"], mode)] = cnt.get((r["lang"], mode), 0) + 1
for lang, (qs, a) in IDENT.items():
    for _ in range(60):
        out.append(render([{"role": "user", "content": rng.choice(qs)}, {"role": "assistant", "content": a}], lang, "instruct"))
rng.shuffle(out)
SFT = f"{WORK}/sft_2b.jsonl"
with open(SFT, "w", encoding="utf-8") as f:
    for r in out:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(out), "examples ->", SFT)
print({f"{l}/{m}": n for (l, m), n in sorted(cnt.items())})
print(out[0]["prompt"][-300:] + out[0]["completion"][:300])

In [ ]:
# 6) Fuse our tokenizer into Sarvam-1's (only where ours is cheaper) + chat/think tokens (~10 min)
fc = {"base": BASE, "our_tok": found["indicmaxxer-48k.json"], "heldout": found["heldout.json"], "extra_jsonl": ULT,
      "min_freq": 8, "scan_top": 30000, "quota": FUSE_QUOTA, "chat_template": CHAT_TEMPLATE,
      "out_dir": FUSED, "report": f"{REP}/fusion_report.json"}
json.dump(fc, open("fuse_config.json", "w"), ensure_ascii=False)
if not os.path.exists(f"{FUSED}/config.json"):
    run([sys.executable, "imx2b_fuse.py", "fuse_config.json"])
assert os.path.exists(f"{FUSED}/config.json"), "fusion did not produce a model"
print(json.dumps(json.load(open(f"{REP}/fusion_report.json", encoding="utf-8")), ensure_ascii=False, indent=1)[:2500])

In [ ]:
# 7) LoRA fine-tune (live log): LoRA + only the NEW token rows. Out of memory -> max_len 768, lora_r 16.
old_vocab = json.load(open(f"{REP}/fusion_report.json"))["vocab_old"]
tc = {"model_dir": FUSED, "data": SFT, "state_path": STATE, "out_dir": OUT, "minutes": TRAIN_MIN, "max_len": 1024,
      "old_vocab": old_vocab,
      "micro_b": 1, "accum": 16, "lora_r": 32, "lr": 2e-4, "emb_lr": 5e-5, "warmup": 20, "log_every": 5,
      "ckpt_every_min": 15}
json.dump(tc, open("train_config.json", "w"))
n = torch.cuda.device_count() if USE_BOTH_GPUS else 1
if n > 1:
    run(["torchrun", "--standalone", f"--nproc_per_node={n}", "imx2b_train.py", "train_config.json"])
else:
    run([sys.executable, "imx2b_train.py", "train_config.json"])
assert os.path.exists(STATE), "training saved no state"

In [ ]:
# 8) Merge LoRA + trained embeddings into the full model -> IndicMaxxer-2B (~3 min)
torch.cuda.empty_cache()
run([sys.executable, "imx2b_merge.py", "train_config.json"])
print(os.listdir(OUT))

In [ ]:
# 8) Quick test: /instruct and /reasoning in 5 styles + native multi-pass reasoning (self-consistency)
import re, collections
from transformers import AutoTokenizer, AutoModelForCausalLM
assert os.path.exists(f"{OUT}/config.json"), f"{OUT} missing: the training or merge cell failed, see its output"
tk = AutoTokenizer.from_pretrained(OUT)
try:
    lm = AutoModelForCausalLM.from_pretrained(OUT, dtype=torch.float16).to("cuda:0").eval()
except TypeError:
    lm = AutoModelForCausalLM.from_pretrained(OUT, torch_dtype=torch.float16).to("cuda:0").eval()
def ask(user, mode="instruct", lang="en", max_new=300, temp=0.0, history=()):
    msgs = [{"role": "system", "content": SYS.get(lang, SYS["en"])}] + list(history) + \
           [{"role": "user", "content": f"{user} /{mode}"}]
    text = tk.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = tk(text, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda:0")
    with torch.no_grad():
        out = lm.generate(ids, max_new_tokens=max_new, do_sample=temp > 0, temperature=max(temp, 1e-5), top_p=0.9,
                          repetition_penalty=1.1, eos_token_id=[tk.convert_tokens_to_ids("<|im_end|>"), tk.eos_token_id],
                          pad_token_id=tk.pad_token_id)
    text = tk.decode(out[0, ids.shape[1]:], skip_special_tokens=False).replace("<|im_end|>", "")
    think, ans = (text.split("</think>", 1) + [""])[:2] if "</think>" in text else ("", text)
    return think.replace("<think>", "").strip(), ans.strip(), out.shape[1] - ids.shape[1]
def last_num(s):
    nums = re.findall(r"-?\d+(?:\.\d+)?", s.replace(",", ""))
    return nums[-1] if nums else None
def reason_k(user, lang, k=3):
    """Think k times natively (sampled), majority vote on the final number."""
    runs = [ask(user, "reasoning", lang, 400, temp=0.7) for _ in range(k)]
    votes = collections.Counter(last_num(a) for _, a, _ in runs if last_num(a))
    return runs, (votes.most_common(1)[0][0] if votes else None), sum(n for _, _, n in runs)
tests = {"kn": "ನನ್ನ ಆರ್ಡರ್ ಇನ್ನೂ ಬಂದಿಲ್ಲ, ಏನು ಮಾಡಬೇಕು?", "hi": "मेरा रिफंड कब तक आएगा?",
         "en": "hii, who are you?", "kanglish": "naale meeting ide, enu prepare maadli?",
         "hinglish": "kal interview hai, kuch tips do"}
res = {}
for lang, q in tests.items():
    th, a, n = ask(q, "instruct", lang)
    res[f"{lang}/instruct"] = a
    print(f"\n[{lang} /instruct] {n} tok\n{a[:400]}")
q = "ರವಿ ಬಳಿ 500 ರೂಪಾಯಿ ಇತ್ತು. ಅವನು 120 ರೂಪಾಯಿಗೆ ಪುಸ್ತಕ ಮತ್ತು 80 ರೂಪಾಯಿಗೆ ಪೆನ್ನು ಖರೀದಿಸಿದನು. ಎಷ್ಟು ಉಳಿಯಿತು?"
th, a, n = ask(q, "reasoning", "kn", 400)
print(f"\n[kn /reasoning] {n} tok\nTHINK: {th[:500]}\nANSWER: {a[:300]}")
runs, vote, ntok = reason_k(q, "kn", 3)
print(f"\n[kn /reasoning x3 self-consistency] vote = {vote} (gold 300), total {ntok} tokens")
res["kn_math"] = {"single": a, "vote": vote}
json.dump(res, open(f"{REP}/test_samples.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)